# Chapter 7 — Message Batches API · Practice

**Goal:** build every piece of a production batch pipeline by hand (payloads, `custom_id` design, crash-safe submit and polling, result triage, resubmitting only the failures, SLA and cost math) and finish with a batch extraction pipeline over ~100 documents.

- Theory: [README.md](README.md) (read section 7.x before running the matching section here)
- Estimated time: 2–3 hours
- Estimated API cost: **$0** for everything offline (the default). The optional **(live)** cells cost **< $0.01** (3 tiny requests on Claude Haiku 4.5 at batch prices, plus a canceled batch). The mini-project with `LIVE = True` costs roughly **$0.05–$0.15** on the default model.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`, `.env` with `ANTHROPIC_API_KEY`), [Chapter 1](../01-claude-api-fundamentals/README.md) (request shape, `stop_reason`, `usage`), [Chapter 2](../02-tool-use/README.md) (client vs server tools, structured outputs), [Chapter 6](../06-prompt-engineering/README.md) (validation and retry with feedback)

How to read the cells:

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key, instantly. Uses `FakeBatchAPI`: a **real** `anthropic.Anthropic` client whose HTTP layer is answered by a local simulator of the Batches API, with a virtual clock (a "24-hour" batch takes milliseconds). |
| **(live)** | Calls the real API and costs a fraction of a cent. Skip it if you have no key; nothing offline depends on it. Real batches can take minutes (rarely hours), so live polling cells give up politely after a while and can be re-run later. |

In [ ]:
import hashlib
import json
import math
import os
import pathlib
import random
import re
import tempfile
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any, Callable, Literal, Protocol

import anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages import MessageBatch, MessageBatchIndividualResponse
from anthropic.types.messages.batch_create_params import Request
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, ValidationError

load_dotenv(find_dotenv())

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
DEMO_MODEL = os.getenv("CLAUDE_BATCH_DEMO_MODEL", "claude-haiku-4-5")   # cheapest model for the tiny live demos
FALLBACK_MODEL = os.getenv("CLAUDE_FALLBACK_MODEL", "claude-haiku-4-5")  # where refused items get a second try

# Only the (live) cells use `client`; offline cells use FakeBatchAPI(...).client
client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") else None
print("anthropic", anthropic.__version__, "| MODEL =", MODEL, "| DEMO_MODEL =", DEMO_MODEL,
      "| live client:", "ready" if client else "none (offline only)")

In [ ]:
# Live cells need a key. If this fails you can still run every (offline) cell below.
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"

### Offline toolkit (run this once)

`FakeBatchAPI` simulates the six batch endpoints (create, retrieve, list, results, cancel, delete) plus `messages.create` and `messages.count_tokens`, following the documented rules:

- `request_counts.processing` holds every request until the batch ends; the four outcome counters stay **0** until then.
- Results come back as a `.jsonl` stream in **shuffled** order.
- `custom_id` must match `^[a-zA-Z0-9_-]{1,64}$` and be unique; a batch holds at most 100,000 requests.
- `stream`, `speed`, `fallbacks` and `max_tokens: 0` come back as per-item `errored` results (the docs document this per-item behavior for `fallbacks`; for the other three they only say "a validation error", and batch validation is reported when the batch ends, so the simulator models all four the same way); a prompt over the (scaled-down) context limit errors with `prompt is too long`.
- Cancel moves a batch to `canceling`; requests that had not run become `canceled`. Batches that take longer than 24 hours leave the rest `expired`.

A **responder** function plays the model: `responder(custom_id, params, attempt) -> {"message": ...} | {"error": (type, msg)} | {"expired": True}`. You will plug in different responders per section. You do not need to read the toolkit code to follow along, but it is short if you are curious.

In [ ]:
import json
import math
import random
import re
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from itertools import count
from typing import Any, Callable

import anthropic
import httpx2

_ids = count(1)
CUSTOM_ID_RE = re.compile(r"^[a-zA-Z0-9_-]{1,64}$")
REJECTED_IN_BATCH = ("stream", "speed", "fallbacks")


def _iso(t: datetime | None) -> str | None:
    return None if t is None else t.isoformat().replace("+00:00", "Z")


def approx_tokens(text: str) -> int:
    """Rough stand-in for the tokenizer (~4 characters per token). Only the fake uses it."""
    return max(1, math.ceil(len(text) / 4))


def _all_text(params: dict) -> str:
    """Concatenate every text the model would read (system + messages), for token estimates."""
    parts = []
    system = params.get("system")
    if isinstance(system, str):
        parts.append(system)
    elif isinstance(system, list):
        parts += [b.get("text", "") for b in system]
    for m in params.get("messages", []):
        c = m.get("content")
        if isinstance(c, str):
            parts.append(c)
        else:
            parts += [b.get("text", "") if isinstance(b, dict) else "" for b in c]
    return "\n".join(parts)


def message(text: str = "", *, stop_reason: str = "end_turn", content: list | None = None,
            input_tokens: int = 100, output_tokens: int | None = None, model: str = "fake-model",
            stop_details: dict | None = None, cache_read: int = 0, cache_write: int = 0) -> dict:
    """A Messages API response body in the exact wire format."""
    blocks = content if content is not None else ([{"type": "text", "text": text}] if text else [])
    out = output_tokens if output_tokens is not None else approx_tokens(json.dumps(blocks))
    return {
        "id": f"msg_fake{next(_ids):05d}", "type": "message", "role": "assistant", "model": model,
        "content": blocks, "stop_reason": stop_reason, "stop_sequence": None, "stop_details": stop_details,
        "usage": {"input_tokens": input_tokens, "output_tokens": out,
                  "cache_creation_input_tokens": cache_write, "cache_read_input_tokens": cache_read},
    }


def default_responder(custom_id: str, params: dict, attempt: int) -> dict:
    """The simplest fake model: answer every request with a short text."""
    text = _all_text(params)
    return {"message": message(f"(fake answer for {custom_id})", input_tokens=approx_tokens(text),
                               model=params.get("model", "fake-model"))}


@dataclass
class _Batch:
    id: str
    requests: list[dict]
    created_at: datetime
    duration: timedelta
    headers: dict
    status: str = "in_progress"
    cancel_initiated_at: datetime | None = None
    ended_at: datetime | None = None
    results: list[dict] = field(default_factory=list)


class FakeBatchAPI:
    """Offline stand-in for the Message Batches API (plus messages.create and count_tokens).

    `.client` is a REAL anthropic.Anthropic client whose HTTP transport is answered here, so the
    SDK's own methods, pagination and typed models (MessageBatch, MessageBatchIndividualResponse...)
    run for real. Time is virtual: call `.sleep(seconds)` to move the clock forward.
    """

    BASE = "https://api.anthropic.com"

    def __init__(self, responder: Callable[[str, dict, int], dict] = default_responder, *,
                 duration_minutes: float = 50, context_limit: int = 1_000_000, seed: int = 7,
                 start: datetime = datetime(2026, 10, 3, 8, 0, tzinfo=timezone.utc)):
        self.responder = responder
        self.duration = timedelta(minutes=duration_minutes)  # simulated processing time per batch
        self.context_limit = context_limit                   # scaled down in demos to make "too long" cheap
        self.rng = random.Random(seed)
        self.now = start
        self.batches: dict[str, _Batch] = {}
        self.attempts: dict[str, int] = {}                   # custom_id -> times it was actually processed
        self.log: list[str] = []                             # "METHOD /path" for every HTTP call
        self.creates = 0
        self.client = anthropic.Anthropic(
            api_key="offline", max_retries=0,
            http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(self._handle)),
        )

    # ---------------------------------------------------------------- virtual clock
    def sleep(self, seconds: float) -> None:
        self.now += timedelta(seconds=seconds)

    # ---------------------------------------------------------------- HTTP router
    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        path, method = request.url.path, request.method
        self.log.append(f"{method} {path}")
        body = json.loads(request.content) if request.content else {}
        m = re.fullmatch(r"/v1/messages/batches/([^/]+)(/results|/cancel)?", path)
        if path == "/v1/messages/batches" and method == "POST":
            return self._create(body, dict(request.headers))
        if path == "/v1/messages/batches" and method == "GET":
            return self._list(dict(request.url.params))
        if path == "/v1/messages" and method == "POST":
            return self._sync(body)
        if path == "/v1/messages/count_tokens" and method == "POST":
            return httpx2.Response(200, json={"input_tokens": approx_tokens(_all_text(body))})
        if m:
            batch = self.batches.get(m.group(1))
            if batch is None:
                return self._error(404, "not_found_error", f"No batch with id {m.group(1)}")
            self._advance(batch)
            if m.group(2) == "/results":
                if batch.status != "ended":
                    return self._error(400, "invalid_request_error", "Batch has not ended yet")
                lines = [json.dumps(r) for r in batch.results]
                return httpx2.Response(200, content=("\n".join(lines) + "\n").encode(),
                                       headers={"content-type": "application/binary"})
            if m.group(2) == "/cancel" and method == "POST":
                if batch.status == "in_progress":
                    batch.status, batch.cancel_initiated_at = "canceling", self.now
                return httpx2.Response(200, json=self._view(batch))
            if method == "DELETE":
                if batch.status != "ended":
                    return self._error(400, "invalid_request_error",
                                       "Batch is still processing; cancel it first, then delete once it has ended")
                del self.batches[batch.id]
                return httpx2.Response(200, json={"id": batch.id, "type": "message_batch_deleted"})
            return httpx2.Response(200, json=self._view(batch))
        return self._error(404, "not_found_error", f"fake has no route for {method} {path}")

    @staticmethod
    def _error(status: int, etype: str, msg: str) -> httpx2.Response:
        return httpx2.Response(status, json={"type": "error", "error": {"type": etype, "message": msg}})

    # ---------------------------------------------------------------- endpoints
    def _create(self, body: dict, headers: dict) -> httpx2.Response:
        reqs = body.get("requests", [])
        if not reqs or len(reqs) > 100_000:
            return self._error(400, "invalid_request_error", "A batch needs 1 to 100,000 requests")
        ids = [r.get("custom_id", "") for r in reqs]
        bad = [i for i in ids if not CUSTOM_ID_RE.match(i)]
        if bad:
            return self._error(400, "invalid_request_error", f"custom_id must match ^[a-zA-Z0-9_-]{{1,64}}$: {bad[:3]}")
        if len(set(ids)) != len(ids):
            return self._error(400, "invalid_request_error", "custom_id values must be unique within a batch")
        self.creates += 1
        bid = f"msgbatch_fake{next(_ids):05d}"
        batch = _Batch(bid, reqs, self.now, self.duration, headers)
        self.batches[bid] = batch
        return httpx2.Response(200, json=self._view(batch))

    def _list(self, params: dict) -> httpx2.Response:
        data = [self._view(b) for b in sorted(self.batches.values(), key=lambda b: b.created_at, reverse=True)]
        limit = int(params.get("limit", 20))
        page = data[:limit]
        return httpx2.Response(200, json={"data": page, "has_more": False,
                                          "first_id": page[0]["id"] if page else None,
                                          "last_id": page[-1]["id"] if page else None})

    def _sync(self, body: dict) -> httpx2.Response:
        """messages.create(): synchronous validation errors come back in seconds, as HTTP 400."""
        problem = self._invalid(body, in_batch=False)
        if problem:
            return self._error(400, "invalid_request_error", problem)
        out = self.responder("sync", body, 0)
        if "error" in out:
            etype, msg = out["error"]
            return self._error(529 if etype == "overloaded_error" else 400, etype, msg)
        if "message" not in out:
            return self._error(500, "api_error", "fake responder returned no message")
        return httpx2.Response(200, json=out["message"])

    def _invalid(self, params: dict, in_batch: bool) -> str | None:
        if in_batch:
            for key in REJECTED_IN_BATCH:
                if key in params:
                    return f"{key}: Extra inputs are not permitted in a Message Batch request"
        mt = params.get("max_tokens")
        if not isinstance(mt, int) or mt < 1:
            return "max_tokens: must be at least 1" + (" (max_tokens: 0 is not supported in batches)" if in_batch else "")
        if approx_tokens(_all_text(params)) > self.context_limit:
            return f"prompt is too long: {approx_tokens(_all_text(params))} tokens > {self.context_limit} maximum"
        return None

    # ---------------------------------------------------------------- simulated processing
    def _advance(self, batch: _Batch) -> None:
        if batch.status == "ended":
            return
        elapsed = self.now - batch.created_at
        day = timedelta(hours=24)
        if batch.status == "canceling":
            if self.now - batch.cancel_initiated_at < timedelta(seconds=30):
                return                                       # cancellation takes a moment to take effect
            done = int(len(batch.requests) * min(1.0, (batch.cancel_initiated_at - batch.created_at) / batch.duration))
            self._finish(batch, processed=done, leftover="canceled")
        elif elapsed >= min(batch.duration, day):
            done = len(batch.requests) if batch.duration <= day else int(len(batch.requests) * (day / batch.duration))
            self._finish(batch, processed=done, leftover="expired")

    def _finish(self, batch: _Batch, processed: int, leftover: str) -> None:
        results = []
        for i, req in enumerate(batch.requests):
            cid, params = req["custom_id"], req["params"]
            if i >= processed:
                results.append({"custom_id": cid, "result": {"type": leftover}})
                continue
            problem = self._invalid(params, in_batch=True)
            if problem:
                results.append({"custom_id": cid, "result": {"type": "errored", "error": {
                    "type": "error", "error": {"type": "invalid_request_error", "message": problem}}}})
                continue
            attempt = self.attempts.get(cid, 0)
            self.attempts[cid] = attempt + 1
            out = self.responder(cid, params, attempt)
            if out.get("expired"):
                results.append({"custom_id": cid, "result": {"type": "expired"}})
            elif "error" in out:
                etype, msg = out["error"]
                results.append({"custom_id": cid, "result": {"type": "errored", "error": {
                    "type": "error", "error": {"type": etype, "message": msg}}}})
            else:
                results.append({"custom_id": cid, "result": {"type": "succeeded", "message": out["message"]}})
        self.rng.shuffle(results)  # results are NOT returned in request order
        batch.results = results
        batch.status = "ended"
        batch.ended_at = self.now

    def _view(self, b: _Batch) -> dict:
        counts = {"processing": 0, "succeeded": 0, "errored": 0, "canceled": 0, "expired": 0}
        if b.status == "ended":
            for r in b.results:
                counts[r["result"]["type"]] += 1
        else:
            counts["processing"] = len(b.requests)  # the outcome counters stay 0 until the batch ends
        return {
            "id": b.id, "type": "message_batch", "processing_status": b.status, "request_counts": counts,
            "created_at": _iso(b.created_at), "expires_at": _iso(b.created_at + timedelta(hours=24)),
            "ended_at": _iso(b.ended_at), "cancel_initiated_at": _iso(b.cancel_initiated_at),
            "archived_at": None,
            "results_url": f"{self.BASE}/v1/messages/batches/{b.id}/results" if b.status == "ended" else None,
        }


print("offline toolkit ready: FakeBatchAPI, message(), approx_tokens()")

## 7.1 What the Message Batches API is

A batch is a list of ordinary Messages API requests, each paired with your own `custom_id`, processed asynchronously for **50% of the standard price**, with **no latency SLA** (most finish within an hour, the ceiling is 24 hours). Limits: 100,000 requests or 256 MB per batch; results downloadable for 29 days after creation. [Theory](README.md#71-what-the-message-batches-api-is)

### Build typed requests and check the size limits (offline)

In [ ]:
DOCS_71 = {
    "inv-00001": "Invoice INV-00001 from Acme Supplies. Widgets 120.00 EUR. Total due 120.00 EUR.",
    "inv-00002": "Invoice INV-00002 from Globex. Consulting 2,400.00 USD. Total 2,400.00 USD.",
    "inv-00003": "Invoice INV-00003 from Initech. Licences 980.50 GBP. Amount due 980.50 GBP.",
}
SYSTEM_71 = "Extract the vendor, currency and total from the invoice. Reply with JSON only."


def extraction_request(custom_id: str, text: str, model: str = MODEL, max_tokens: int = 256) -> Request:
    """One batch entry: your custom_id + exactly the params you would pass to messages.create()."""
    return Request(
        custom_id=custom_id,
        params=MessageCreateParamsNonStreaming(
            model=model,
            max_tokens=max_tokens,
            system=SYSTEM_71,
            messages=[{"role": "user", "content": text}],
        ),
    )


requests_71 = [extraction_request(cid, text) for cid, text in DOCS_71.items()]
print(json.dumps(requests_71[0], indent=1))   # Request and MessageCreateParamsNonStreaming are TypedDicts: plain dicts

MAX_REQUESTS, MAX_BYTES = 100_000, 256 * 1024 * 1024


def body_bytes(requests: list[Request]) -> int:
    return len(json.dumps({"requests": requests}).encode())


def split_into_batches(requests: list[Request], max_requests: int = MAX_REQUESTS,
                       max_bytes: int = MAX_BYTES) -> list[list[Request]]:
    """Greedy packing that respects both limits (whichever is reached first)."""
    batches, current, size = [], [], len(b'{"requests": []}')
    for r in requests:
        r_bytes = len(json.dumps(r).encode()) + 2
        if current and (len(current) == max_requests or size + r_bytes > max_bytes):
            batches.append(current)
            current, size = [], len(b'{"requests": []}')
        current.append(r)
        size += r_bytes
    return batches + ([current] if current else [])


print(f"\n{len(requests_71)} requests, {body_bytes(requests_71):,} bytes")
# What if each request carried a 40 KB document and we had 10,000 of them?
big = [extraction_request(f"doc-{i:05d}", "x" * 40_000) for i in range(10_000)]
parts = split_into_batches(big)
print(f"10,000 x 40 KB documents = {body_bytes(big) / 2**20:,.0f} MB -> {len(parts)} batches of",
      [len(p) for p in parts], "requests")

> **Observe:** the request list is about 384 MB, so it is the **256 MB** limit, not the 100,000-request limit, that forces a split here. For large documents, upload them once with the Files API and reference them by `file_id` to keep the request body small (README 7.3).

### The lifecycle through the SDK (offline)

In [ ]:
def show_batch(b: MessageBatch) -> None:
    c = b.request_counts
    print(f"{b.id}  status={b.processing_status:11}  processing={c.processing} succeeded={c.succeeded} "
          f"errored={c.errored} canceled={c.canceled} expired={c.expired}")
    print(f"   created_at={b.created_at:%Y-%m-%d %H:%M}  expires_at={b.expires_at:%Y-%m-%d %H:%M}  "
          f"ended_at={b.ended_at and f'{b.ended_at:%H:%M}'}  results_url={'set' if b.results_url else None}")


fake = FakeBatchAPI(duration_minutes=50)          # this simulated batch takes 50 minutes
api = fake.client                                 # a real anthropic.Anthropic client, fake transport

batch = api.messages.batches.create(requests=requests_71)
show_batch(batch)
fake.sleep(20 * 60)                               # virtual clock: 20 minutes later
show_batch(api.messages.batches.retrieve(batch.id))
fake.sleep(40 * 60)
show_batch(api.messages.batches.retrieve(batch.id))

print("\nResults (streamed .jsonl, typed as", MessageBatchIndividualResponse.__name__, "):")
for item in api.messages.batches.results(batch.id):
    print(f"   {item.custom_id}: {item.result.type}")

print("\nAll batches in the workspace:", [b.id for b in api.messages.batches.list(limit=20)])
print("HTTP calls the SDK made:", *fake.log, sep="\n   ")

> **Observe:**
> - After 20 minutes `succeeded` is still **0** even though work is happening: outcome counters are filled only when the batch ends.
> - `results_url` appears only after the batch ends, and `results()` first calls `retrieve()` to find it (look at the HTTP log).
> - The results are not in the order you submitted them.

> **Try it:** create `FakeBatchAPI(duration_minutes=26 * 60)` (a batch that would need 26 hours) and repeat the cell with `fake.sleep(25 * 3600)`. Which result type do the unprocessed requests get?

### A real 3-request batch (live, ~$0.001)

In [ ]:
# (live) Submit 3 tiny requests on the cheapest model. Section 7.5 polls it and streams the results.
live_requests = [extraction_request(cid, text, model=DEMO_MODEL, max_tokens=100) for cid, text in DOCS_71.items()]
live_batch = client.messages.batches.create(requests=live_requests)
LIVE_BATCH_ID = live_batch.id
show_batch(live_batch)

## 7.2 Batch vs synchronous: matching the API to latency requirements

The deciding question is **"is anyone (a person or a pipeline stage) blocked waiting for this answer?"**, not "how many requests?". Blocking work stays synchronous; scheduled, latency-tolerant work goes to batches, provided the SLA leaves room for the 24-hour worst case and your code does not need to run tools mid-request. [Theory](README.md#72-batch-vs-synchronous-matching-the-api-to-latency-requirements)

In [ ]:
@dataclass
class Workload:
    name: str
    blocking: bool                       # is a person or pipeline stage waiting on the answer?
    sla_hours: float | None = None       # deadline measured from arrival; None = no deadline
    client_tools_midrequest: bool = False
    zdr_required: bool = False


def choose_api(w: Workload, buffer_hours: float = 2.0) -> tuple[str, str]:
    if w.blocking:
        return "sync", "someone is blocked; batches have no latency SLA"
    if w.zdr_required:
        return "sync", "the Batches API is not ZDR-eligible (stores data up to 29 days)"
    if w.sla_hours is not None and w.sla_hours < 24 + buffer_hours:
        return "sync", f"SLA {w.sla_hours:g} h < 24 h worst case + {buffer_hours:g} h buffer"
    if w.client_tools_midrequest:
        return "sync (or restructure)", "a batch request cannot run YOUR tools and continue"
    return "batch", "latency-tolerant: 50% off every token"


WORKLOADS = [
    Workload("Pre-merge PR check that blocks merging", blocking=True),
    Workload("Interactive support agent", blocking=True),
    Workload("Overnight tech-debt report for the 09:00 stand-up", blocking=False, sla_hours=36),
    Workload("Weekly security audit of the codebase", blocking=False, sla_hours=7 * 24),
    Workload("10,000 non-urgent documents overnight", blocking=False, sla_hours=48),
    Workload("Eval suite of 5,000 cases before a model upgrade", blocking=False),
    Workload("Iterative review that fetches files via read_file", blocking=False, sla_hours=72,
             client_tools_midrequest=True),
    Workload("Ticket triage with a 4-hour SLA", blocking=False, sla_hours=4),
    Workload("Nightly HR-case summaries, customer has ZDR", blocking=False, sla_hours=48, zdr_required=True),
]
for w in WORKLOADS:
    api_choice, why = choose_api(w)
    print(f"{w.name:52} -> {api_choice:22} {why}")

Why "batch it, with a synchronous fallback if it takes too long" is the wrong answer for a blocking check:

In [ ]:
def blocked_minutes(fallback_timeout_min: float, sync_minutes: float = 1.5, batch_minutes: float | None = None) -> float:
    """How long a developer waits under 'batch first, then sync after a timeout'."""
    if batch_minutes is not None and batch_minutes <= fallback_timeout_min:
        return batch_minutes
    return fallback_timeout_min + sync_minutes          # waited for the timeout, THEN the sync call


for timeout in (10, 30, 60):
    print(f"timeout {timeout:>2} min: slow batch -> developer waits {blocked_minutes(timeout):5.1f} min "
          f"(plain sync: 1.5 min), and you may pay for both calls")

> **Observe:** the fallback never removes the wait; it only caps it at "timeout + sync time", and when the batch is slow you pay for both paths.

> **Try it:** add a `Workload` for "real-time fraud flag on a payment" and for "deep analysis that already posts its result to the PR an hour later". Predict the answer before you run it. Then remember the vocabulary trap from the README: Claude Code has no `--batch` flag, and the Academy's "batch tool" is a tool-design pattern, not this API.

## 7.3 What can and cannot be batched

Almost any Messages request can be batched (vision, PDFs, pre-built multi-turn histories, thinking, structured outputs, tool definitions and **all server tools**). Rejected: `stream`, `speed`, `max_tokens: 0` and the beta `fallbacks`. The real limitation: a batch request cannot pause for **your** code to run a client tool and then continue. [Theory](README.md#73-what-can-and-cannot-be-batched)

### A pre-flight linter for batch params (offline)

In [ ]:
SYNC_MAX_TOKENS, EXTENDED_MAX_TOKENS = 128_000, 300_000
EXTENDED_OUTPUT_BETA = "output-300k-2026-03-24"
EXTENDED_OUTPUT_MODELS = {"claude-opus-5-5", "claude-opus-5", "claude-opus-4-8", "claude-opus-4-7",
                          "claude-opus-4-6", "claude-sonnet-5-5", "claude-sonnet-5", "claude-sonnet-4-6"}


def batch_param_problems(params: dict, betas: tuple[str, ...] = ()) -> list[str]:
    """Errors (would fail) and warnings (would 'succeed' but be unusable) for one batch request."""
    out = []
    if params.get("stream"):
        out.append("ERROR stream: results come back as one file, not a stream")
    if "speed" in params:
        out.append("ERROR speed: fast mode tunes synchronous latency; not supported in batches")
    if "fallbacks" in params:
        out.append("ERROR fallbacks: server-side refusal fallback is not supported (item comes back errored)")
    mt = params.get("max_tokens", 0)
    if mt < 1:
        out.append("ERROR max_tokens must be >= 1 (max_tokens: 0 cache pre-warming is not supported)")
    elif mt > SYNC_MAX_TOKENS:
        if EXTENDED_OUTPUT_BETA not in betas:
            out.append(f"ERROR max_tokens {mt:,} > 128K needs the {EXTENDED_OUTPUT_BETA} beta (see 7.10)")
        elif params.get("model") not in EXTENDED_OUTPUT_MODELS:
            out.append(f"ERROR extended output is not available on {params.get('model')}")
        if mt > EXTENDED_MAX_TOKENS:
            out.append("ERROR max_tokens above 300K")
    for tool in params.get("tools", []):
        if tool.get("type") in (None, "custom"):           # a client tool: YOUR code would have to run it
            out.append(f"WARN client tool '{tool['name']}': if Claude calls it, the request just ends with "
                       "stop_reason 'tool_use'; nothing in the batch can run it and continue")
    return out


base = {"model": MODEL, "max_tokens": 1024, "messages": [{"role": "user", "content": "hi"}]}
CASES_73 = {
    "plain request": base,
    "streaming": {**base, "stream": True},
    "fast mode": {**base, "speed": "fast"},
    "cache pre-warm": {**base, "max_tokens": 0},
    "refusal fallbacks": {**base, "fallbacks": "default"},
    "200K output, no beta": {**base, "max_tokens": 200_000},
    "web search (server tool)": {**base, "tools": [{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}]},
    "read_file (client tool)": {**base, "tools": [{"name": "read_file", "description": "Read a repo file.",
                                                  "input_schema": {"type": "object", "properties": {"path": {"type": "string"}}}}]},
    "pre-built multi-turn": {**base, "messages": [
        {"role": "user", "content": "Classify: 'refund please'"}, {"role": "assistant", "content": "billing"},
        {"role": "user", "content": "Classify: 'app crashes on login'"}]},
}
for label, params in CASES_73.items():
    print(f"{label:26} {batch_param_problems(params) or 'OK'}")

> **Observe:** the pre-built multi-turn conversation is fine (you wrote the history in advance; it is still one request, one response). Server tools pass; a client tool only earns a warning, because the definition is accepted but the round trip cannot happen inside the batch.

### What actually comes back: client tool vs server tool (offline)

We script the model so you can see both shapes in real SDK types: a client `tool_use` stop that the batch cannot finish, and a server-tool `pause_turn` that you continue with a follow-up request.

In [ ]:
def responder_73(custom_id: str, params: dict, attempt: int) -> dict:
    if custom_id == "review-base-py":       # Claude wants YOUR read_file tool
        return {"message": message(content=[
            {"type": "text", "text": "I need the base class first."},
            {"type": "tool_use", "id": "toolu_01", "name": "read_file", "input": {"path": "src/base.py"}}],
            stop_reason="tool_use")}
    if custom_id == "enrich-acme":          # a long server-side web search loop hit the per-turn limit
        return {"message": message(content=[
            {"type": "text", "text": "Searching for Acme's latest filings."},
            {"type": "server_tool_use", "id": "srvtoolu_01", "name": "web_search", "input": {"query": "Acme Corp 10-K 2026"}}],
            stop_reason="pause_turn")}
    return default_responder(custom_id, params, attempt)


fake = FakeBatchAPI(responder_73, duration_minutes=30)
api = fake.client
read_file_tool = CASES_73["read_file (client tool)"]["tools"][0]
web_search_tool = CASES_73["web search (server tool)"]["tools"][0]
originals = {
    "review-base-py": {**base, "tools": [read_file_tool], "messages": [{"role": "user", "content": "Review src/models.py"}]},
    "enrich-acme": {**base, "tools": [web_search_tool], "messages": [{"role": "user", "content": "Find Acme's 2026 revenue."}]},
}
b = api.messages.batches.create(requests=[Request(custom_id=k, params=v) for k, v in originals.items()])
fake.sleep(3600)

follow_ups = []
for item in api.messages.batches.results(b.id):
    msg = item.result.message
    print(f"{item.custom_id:15} stop_reason={msg.stop_reason:10} blocks={[blk.type for blk in msg.content]}")
    if msg.stop_reason == "pause_turn":
        # Continue: same request + the paused assistant content appended (batch or synchronous)
        p = originals[item.custom_id]
        follow_ups.append(Request(custom_id=item.custom_id, params={
            **p, "messages": p["messages"] + [{"role": "assistant", "content": [blk.to_dict() for blk in msg.content]}]}))
    elif msg.stop_reason == "tool_use":
        print("   -> unfinished: running read_file and continuing needs a NEW request (another wait of up to 24 h)")

print("\nfollow-up request for the paused turn (last message):")
print(json.dumps(follow_ups[0]["params"]["messages"][-1], indent=1))

> **Observe:** the follow-up keeps the **same `custom_id`** (one logical item = one key) and re-sends the assistant content exactly as returned. The batch worker allows more server-tool iterations per turn than a synchronous request, so `pause_turn` should be rare, but your pipeline must still handle it.

> **Try it:** restructure the `review-base-py` request so it can be batched: drop the tool, read `src/base.py` yourself beforehand, and put its contents in the prompt inside `<file path="src/base.py">` tags. That is the "pre-fetch context, then batch a single-turn analysis" pattern from the README.

## 7.4 Designing `custom_id`

`custom_id` is the only link between a result and its source, and results come back unordered: treat it as a primary key. API rules: 1–64 characters of `[a-zA-Z0-9_-]`, unique per batch. Design rules: derive it from your record key, add `-cN` for chunks, hash long or PII-bearing keys, keep it stable across retries, and keep the details in a ledger. [Theory](README.md#74-designing-custom_id)

In [ ]:
CUSTOM_ID_RE = re.compile(r"^[a-zA-Z0-9_-]{1,64}$")


def make_custom_id(doc_key: str, chunk: int | None = None) -> str:
    """Stable, legal, PII-free custom_id for a document (and optional chunk)."""
    base = re.sub(r"[^a-zA-Z0-9_-]", "-", doc_key)
    if len(base) > 48 or base != doc_key:          # long, or contained illegal characters
        base = "d-" + hashlib.sha256(doc_key.encode()).hexdigest()[:20]
    cid = base if chunk is None else f"{base}-c{chunk}"
    assert CUSTOM_ID_RE.match(cid), cid
    return cid


KEYS = ["inv-2026-000417", "contracts/2026/ACME Corp/MSA v3.pdf", "jane.doe@example.com",
        "customer_" + "x" * 70]
for k in KEYS:
    print(f"{k[:40]:42} -> {make_custom_id(k)}")
print("chunk 2 of the first key ->", make_custom_id(KEYS[0], chunk=2))
print("stable across runs:", make_custom_id(KEYS[1]) == make_custom_id(KEYS[1]))

The ledger keeps what the ID must not carry (paths, emails, attempt counters, batch IDs):

In [ ]:
@dataclass
class LedgerEntry:
    custom_id: str
    source: str                        # original key / path (never sent as the ID)
    content_sha256: str
    attempt: int = 0
    batch_ids: list[str] = field(default_factory=list)
    status: str = "pending"


def ledger_for(docs: dict[str, str]) -> dict[str, LedgerEntry]:
    return {make_custom_id(k): LedgerEntry(make_custom_id(k), k, hashlib.sha256(t.encode()).hexdigest()[:12])
            for k, t in docs.items()}


ledger_74 = ledger_for({k: f"text of {k}" for k in KEYS})
for e in ledger_74.values():
    print(e)

### Why matching by position fails (offline)

In [ ]:
fake = FakeBatchAPI(lambda cid, p, a: {"message": message(f"extraction for {cid}")}, duration_minutes=10)
api = fake.client
b = api.messages.batches.create(requests=requests_71)
fake.sleep(900)
results = list(api.messages.batches.results(b.id))

by_position = dict(zip(DOCS_71, results))                    # WRONG: assumes request order
by_custom_id = {r.custom_id: r for r in results}             # RIGHT: primary key
wrong = sum(by_position[k].custom_id != k for k in DOCS_71)
print(f"zip(): {wrong} of {len(DOCS_71)} documents got someone else's result")
print("custom_id: every document matched:", all(by_custom_id[k].custom_id == k for k in DOCS_71))

> **Observe:** with three requests the shuffle may leave one in place by luck; with thousands, `zip()` silently corrupts almost every row, and nothing raises an error.

> **Try it:** change `make_custom_id` so that hashed IDs keep a readable prefix (for example the first 8 safe characters of the key, then the hash). Check that the result is still at most 64 characters and contains no `@`.

## 7.5 Tracking a batch: polling, streaming results, cancel and delete

Persist the batch ID (and the `custom_id`s it contains) **before anything else**, so a crash never leads to a double submission. Poll `retrieve()` with backoff until `processing_status == "ended"`, then **stream** `results()` one line at a time. `cancel()` moves the batch to `canceling` (unsent requests become `canceled`); `delete()` works only after the batch has ended. [Theory](README.md#75-tracking-a-batch-polling-streaming-results-cancel-and-delete)

### Crash-safe submit (offline)

In [ ]:
WORKDIR = pathlib.Path(tempfile.mkdtemp(prefix="ch07-"))


def submit_once(api: anthropic.Anthropic, requests: list[Request], state_path: pathlib.Path) -> str:
    """Never creates the same batch twice: a saved state file means 'resume, don't resubmit'."""
    if state_path.exists():
        return json.loads(state_path.read_text())["batch_id"]
    batch = api.messages.batches.create(requests=requests)
    state_path.write_text(json.dumps({                       # persist IMMEDIATELY after create()
        "batch_id": batch.id,
        "created_at": batch.created_at.isoformat(),
        "custom_ids": [r["custom_id"] for r in requests],
    }))
    return batch.id


fake = FakeBatchAPI(duration_minutes=50)
api = fake.client
state = WORKDIR / "batch_state.json"

first = submit_once(api, requests_71, state)
# ... the orchestrator crashes here and is restarted by the scheduler ...
second = submit_once(api, requests_71, state)
print("batch ids:", first, second, "| batches actually created:", fake.creates)
print("state file:", state.read_text()[:160], "...")

### Polling with backoff (offline, virtual clock)

In [ ]:
def wait_until_ended(api: anthropic.Anthropic, batch_id: str, sleep: Callable[[float], None] = time.sleep,
                     first_delay: float = 30, max_delay: float = 600, max_wait: float = 25 * 3600,
                     log: list | None = None) -> MessageBatch | None:
    """Poll with exponential backoff. Returns the ended batch, or None if max_wait passed."""
    delay, waited = first_delay, 0.0
    while True:
        batch = api.messages.batches.retrieve(batch_id)
        if log is not None:
            log.append((waited, batch.processing_status, batch.request_counts.succeeded))
        if batch.processing_status == "ended":
            return batch                      # request_counts are final now
        if waited >= max_wait:
            return None
        sleep(delay)
        waited += delay
        delay = min(delay * 2, max_delay)


poll_log: list = []
ended = wait_until_ended(api, first, sleep=fake.sleep, log=poll_log)
for waited, status, succeeded in poll_log:
    print(f"t+{waited / 60:6.1f} min  {status:11} succeeded={succeeded}")
print(f"\n{len(poll_log)} status calls with backoff vs {50 * 60 // 10} calls polling every 10 s for the same 50 minutes")

> **Observe:** status calls count against the Batches API's requests-per-minute limit (7.11), and `succeeded=0` during processing is not a failure signal.

### Stream results into a sink, one line at a time (offline)

In [ ]:
sink = WORKDIR / "results.jsonl"
with sink.open("w") as f:
    for item in api.messages.batches.results(first):          # a generator: never the whole file in memory
        row = {"custom_id": item.custom_id, "type": item.result.type}
        if item.result.type == "succeeded":
            row["text"] = item.result.message.content[0].text
            row["usage"] = item.result.message.usage.model_dump(include={"input_tokens", "output_tokens"})
        f.write(json.dumps(row) + "\n")
print(sink.read_text())

### Cancel and delete (offline)

In [ ]:
fake = FakeBatchAPI(duration_minutes=50)
api = fake.client
ten = [extraction_request(f"doc-{i:03d}", f"document {i}") for i in range(10)]
b = api.messages.batches.create(requests=ten)
fake.sleep(20 * 60)                                   # 40% of the way through
show_batch(api.messages.batches.cancel(b.id))

try:
    api.messages.batches.delete(b.id)
except anthropic.BadRequestError as e:
    print("delete while canceling ->", e.status_code, e.body["error"]["message"])

final = wait_until_ended(api, b.id, sleep=fake.sleep, first_delay=10)
show_batch(final)
print("partial results:", Counter(r.result.type for r in api.messages.batches.results(b.id)))
print("deleted:", api.messages.batches.delete(b.id))
print("list after delete:", [x.id for x in api.messages.batches.list()])

> **Observe:** the canceled batch still **ends normally**, with real results for the requests that ran first; the rest are `canceled` and not billed. Collect them before you delete.

> **Try it:** delete `batch_state.json` and call `submit_once` again: now a second batch is created. That is exactly the double payment the state file prevents. If you ever lose an ID, `api.messages.batches.list()` shows every batch in the workspace.

### Poll and stream the live batch from 7.1 (live)

In [ ]:
# (live) Polls for up to ~15 minutes. Most small batches end within minutes; if not, re-run this cell later.
batch_id = globals().get("LIVE_BATCH_ID")
if batch_id is None:
    print("No live batch yet: run the (live) cell in 7.1 first.")
else:
    ended = wait_until_ended(client, batch_id, first_delay=15, max_delay=120, max_wait=15 * 60)
    if ended is None:
        print("Still processing. That is normal (no latency SLA). Re-run this cell later.")
    else:
        show_batch(ended)
        for item in client.messages.batches.results(batch_id):
            if item.result.type == "succeeded":
                m = item.result.message
                print(item.custom_id, m.stop_reason, m.usage.input_tokens, "in /", m.usage.output_tokens, "out |",
                      m.content[0].text[:80].replace("\n", " "))
            else:
                print(item.custom_id, item.result.type)

In [ ]:
# (live) Cancel demo: create, cancel at once, then wait for 'ended' and look at the counts.
b = client.messages.batches.create(requests=[extraction_request(f"cancel-demo-{i}", text, model=DEMO_MODEL, max_tokens=50)
                                             for i, text in enumerate(DOCS_71.values())])
show_batch(client.messages.batches.cancel(b.id))
ended = wait_until_ended(client, b.id, first_delay=5, max_delay=60, max_wait=10 * 60)
if ended:
    show_batch(ended)
    client.messages.batches.delete(b.id)       # clean up stored data once processing has ended
    print("deleted", b.id)

> **Observe:** cancellation may not take effect immediately, so some or all of the three requests may still come back `succeeded`; the rest are `canceled`.

## 7.6 Result types and resubmitting only the failures

Four result types: `succeeded`, `errored`, `canceled`, `expired` (only `succeeded` is billed). **Succeeded is not usable** until you check `stop_reason` (`refusal`, `max_tokens`, `model_context_window_exceeded`, `pause_turn`, `tool_use`) and validate the output. Then resubmit **only** the failed `custom_id`s, each with the modification that fixes its cause. [Theory](README.md#76-result-types-and-resubmitting-only-the-failures)

In [ ]:
class Invoice(BaseModel):
    vendor: str
    invoice_number: str | None          # null = not printed on the document (a valid answer)
    total_cents: int | None
    line_items_cents: list[int]


def ok_json(**kw) -> dict:
    data = {"vendor": "Acme", "invoice_number": "INV-1", "total_cents": 1500, "line_items_cents": [1000, 500]} | kw
    return {"message": message(json.dumps(data))}


SCRIPT_76 = {
    "inv-ok":          ok_json(),
    "inv-no-number":   ok_json(invoice_number=None),
    "inv-no-total":    ok_json(total_cents=None),
    "inv-bad-sum":     ok_json(total_cents=1700),
    "inv-schema":      {"message": message('{"vendor": "Acme", "total_cents": "1,500.00"}')},
    "inv-truncated":   {"message": message('{"vendor": "Acme", "line_items_cents": [1000, 50', stop_reason="max_tokens")},
    "inv-refused":     {"message": message(content=[], stop_reason="refusal", output_tokens=0, stop_details={
                           "type": "refusal", "category": "cyber",
                           "explanation": "This request was declined because it could enable cyber harm."})},
    "inv-too-long":    {"error": ("invalid_request_error", "prompt is too long: 1204331 tokens > 1000000 maximum")},
    "inv-overloaded":  {"error": ("overloaded_error", "Overloaded")},
    "inv-expired":     {"expired": True},
}
fake = FakeBatchAPI(lambda cid, p, a: SCRIPT_76[cid], duration_minutes=40)
api = fake.client
batch_76 = api.messages.batches.create(requests=[extraction_request(cid, "...") for cid in SCRIPT_76])
fake.sleep(3600)
show_batch(api.messages.batches.retrieve(batch_76.id))

A classifier that turns every result into a bucket plus a reason. `semantic` is your own business check.

In [ ]:
Bucket = Literal["ok", "retry_as_is", "fix_then_retry", "fallback_model", "continue_turn", "human"]


def invoice_semantics(inv: Invoice) -> tuple[Bucket, str] | None:
    if inv.total_cents is None:
        return "human", "total not in the document (null): a retry cannot create it"
    if sum(inv.line_items_cents) != inv.total_cents:
        return "fix_then_retry", (f"line items sum to {sum(inv.line_items_cents)} but total_cents is "
                                  f"{inv.total_cents}; re-check each amount")
    return None


def classify(item: MessageBatchIndividualResponse, schema: type[BaseModel],
             semantic: Callable[[Any], tuple[Bucket, str] | None] | None = None) -> tuple[Bucket, str, Any]:
    r = item.result
    if r.type in ("canceled", "expired"):                    # never ran, not billed
        return "retry_as_is", r.type, None
    if r.type == "errored":
        err = r.error.error                                    # ErrorResponse -> typed error object
        if err.type == "invalid_request_error":                # deterministic: fix the request first
            return "fix_then_retry", f"invalid_request: {err.message}", None
        return "retry_as_is", err.type, None                   # api_error / overloaded_error: transient
    msg = r.message
    if msg.stop_reason == "refusal":
        category = msg.stop_details.category if msg.stop_details else None
        return "fallback_model", f"refusal (category={category})", None
    if msg.stop_reason in ("max_tokens", "model_context_window_exceeded"):
        return "fix_then_retry", f"truncated ({msg.stop_reason})", None
    if msg.stop_reason == "pause_turn":
        return "continue_turn", "server-tool loop paused", None
    if msg.stop_reason == "tool_use":
        return "human", "asked for a client tool: redesign the request (7.3)", None
    if msg.stop_reason != "end_turn":
        return "human", f"unexpected stop_reason {msg.stop_reason}", None
    text = "".join(b.text for b in msg.content if b.type == "text")
    try:
        parsed = schema.model_validate_json(text)
    except ValidationError as e:
        first = e.errors()[0]
        return "fix_then_retry", f"schema: {'.'.join(map(str, first['loc'])) or '$'}: {first['msg']}", None
    verdict = semantic(parsed) if semantic else None
    return (*verdict, parsed) if verdict else ("ok", "valid", parsed)


buckets: dict[str, list[tuple[str, str]]] = defaultdict(list)
for item in api.messages.batches.results(batch_76.id):
    bucket, reason, _ = classify(item, Invoice, invoice_semantics)
    buckets[bucket].append((item.custom_id, reason))
for bucket in ("ok", "retry_as_is", "fix_then_retry", "fallback_model", "continue_turn", "human"):
    for cid, reason in sorted(buckets.get(bucket, [])):
        print(f"{bucket:15} {cid:15} {reason}")

> **Observe:** `request_counts` said 7 succeeded, yet only 2 are usable. `inv-no-number` is **ok**: a `null` for a field that is not on the document is a correct answer, not a failure. `inv-no-total` goes to a human because the business needs a total and no retry can invent one.

### Plan the resubmission: only the failures, each with its fix (offline)

In [ ]:
def resubmission_plan(buckets: dict[str, list[tuple[str, str]]], originals: dict[str, dict]) -> tuple[list[Request], list[str]]:
    """Return (requests for a NEW batch containing only fixable failures, ids routed to humans)."""
    new, human = [], [cid for cid, _ in buckets.get("human", [])]
    for bucket, items in buckets.items():
        for cid, reason in items:
            p = originals[cid]
            if bucket == "retry_as_is":
                new.append(Request(custom_id=cid, params=p))                                  # unchanged
            elif bucket == "fallback_model":
                new.append(Request(custom_id=cid, params={**p, "model": FALLBACK_MODEL}))
            elif bucket == "fix_then_retry" and reason.startswith("truncated"):
                new.append(Request(custom_id=cid, params={**p, "max_tokens": p["max_tokens"] * 2}))
            elif bucket == "fix_then_retry" and "prompt is too long" in reason:
                human_text = p["messages"][-1]["content"]
                for i, part in enumerate(["<first half>", "<second half>"]):              # real chunking: mini-project
                    new.append(Request(custom_id=f"{cid}-c{i}", params={
                        **p, "messages": [{"role": "user", "content": f"{human_text} {part}"}]}))
            elif bucket == "fix_then_retry":                                                  # schema / semantic
                new.append(Request(custom_id=cid, params={**p, "messages": p["messages"] + [
                    {"role": "assistant", "content": "(previous answer)"},
                    {"role": "user", "content": f"Validation feedback: {reason}. Return corrected JSON only."}]}))
    return new, human


originals_76 = {cid: extraction_request(cid, "...")["params"] for cid in SCRIPT_76}
retry_requests, to_humans = resubmission_plan(buckets, originals_76)
for r in retry_requests:
    p = r["params"]
    print(f"{r['custom_id']:18} model={p['model']:18} max_tokens={p['max_tokens']:4} turns={len(p['messages'])}")
print("to human review:", to_humans)
print("successes NOT resubmitted:", [cid for cid, _ in buckets["ok"]])

> **Observe:** the new batch has 8 requests for 7 failed documents (one was split in two), and neither success is reprocessed. The chunk IDs extend the parent ID (`inv-too-long-c0`) so their results can be re-assembled.

> **Try it:** add a cap: after 3 attempts (track them in a ledger, not in the ID), route the item to `human` instead of resubmitting. Where would you store the attempt counter so it survives a crash?

### The running example for the rest of the notebook: ~100 synthetic documents and a fake extractor

From here on we use one corpus: invoices, receipts and credit notes with line items and a total. A deterministic **fake extractor** plays the model, so you can see the effect of prompts, chunking and retries without spending anything. Some documents carry a hidden simulation tag (an HTML comment) that makes the fake misbehave once: overloaded, expired, or a dropped line item. To keep failures cheap, the world is **scaled down**: the fake's context window is 3,000 tokens instead of 1M, and requests use `max_tokens=256`.

In [ ]:
VENDORS = ["Acme Supplies", "Globex", "Initech", "Umbrella Retail", "Stark Parts", "Wayne Logistics", "Hooli",
           "Vandelay Imports"]
ITEMS = ["Widget", "Gasket", "Bracket", "Cable", "Sensor", "Valve", "Bearing", "Panel"]
HEADER = {"invoice": "INVOICE", "receipt": "RECEIPT", "credit_note": "CREDIT NOTE"}
SIM_TAGS = {"overloaded": "<!-- sim:overloaded-once -->", "expires": "<!-- sim:expires-once -->",
            "drops_line": "<!-- sim:drops-a-line -->",
            "pentest": "Notes: includes a proof-of-concept exploit snippet from the penetration test."}
FAKE_CONTEXT_LIMIT = 3_000


def make_doc(i: int, doc_type: str, n_lines: int, rng: random.Random, *, with_total: bool = True,
             tags: tuple[str, ...] = ()) -> dict:
    number = f"{doc_type[:3].upper()}-2026-{i:05d}"
    amounts = [rng.randint(100, 50_000) for _ in range(n_lines)]
    lines = [f"{HEADER[doc_type]} {number}", f"Vendor: {VENDORS[i % len(VENDORS)]}", "Currency: EUR"]
    lines += [f"- {ITEMS[j % len(ITEMS)]} #{j + 1} ........ {a / 100:.2f}" for j, a in enumerate(amounts)]
    if with_total:
        lines.append(f"{'Amount paid' if doc_type == 'receipt' else 'TOTAL'}: {sum(amounts) / 100:.2f}")
    lines += [SIM_TAGS[t] for t in tags]
    return {"doc_id": number, "doc_type": doc_type, "text": "\n".join(lines)}


def make_corpus(n: int = 100, seed: int = 42) -> list[dict]:
    rng = random.Random(seed)
    special = {  # index -> (n_lines, with_total, tags): the troublemakers
        3: (450, True, ()), 17: (450, True, ()), 58: (450, True, ()),      # too long for the context window
        8: (180, True, ()), 33: (180, True, ()), 71: (180, True, ()),      # output longer than max_tokens
        12: (5, True, ("overloaded",)), 45: (7, True, ("overloaded",)),
        26: (6, True, ("expires",)), 64: (4, True, ("expires",)),
        39: (3, True, ("pentest",)),
        20: (8, True, ("drops_line",)), 52: (5, True, ("drops_line",)), 90: (9, True, ("drops_line",)),
        77: (6, False, ()), 95: (4, False, ()),                              # total genuinely missing
    }
    docs = []
    for i in range(n):
        doc_type = "receipt" if i % 4 == 1 else "credit_note" if i % 7 == 2 else "invoice"
        if i in (77, 95):
            doc_type = "invoice"
        n_lines, with_total, tags = special.get(i, (rng.randint(2, 12), True, ()))
        docs.append(make_doc(i, doc_type, n_lines, rng, with_total=with_total, tags=tags))
    return docs


def _texts(params: dict, role: str) -> str:
    out = []
    for m in params["messages"]:
        if m["role"] == role:
            c = m["content"]
            out.append(c if isinstance(c, str) else " ".join(b.get("text", "") for b in c if isinstance(b, dict)))
    return "\n".join(out)


def fake_extractor(custom_id: str, params: dict, attempt: int) -> dict:
    """Plays the model. Deterministic, but it has the weaknesses a real pipeline must handle."""
    system = params["system"] if isinstance(params.get("system"), str) else " ".join(
        b["text"] for b in params.get("system", []))
    doc = _texts(params, "user")
    in_batch = custom_id != "sync"
    if in_batch and attempt == 0 and SIM_TAGS["overloaded"] in doc:
        return {"error": ("overloaded_error", "Overloaded")}
    if in_batch and attempt == 0 and SIM_TAGS["expires"] in doc:
        return {"expired": True}
    n_in = approx_tokens(system + doc)
    if "proof-of-concept exploit" in doc and params["model"] == MODEL:   # benign work can trip a classifier
        return {"message": message(content=[], stop_reason="refusal", input_tokens=n_in, output_tokens=0,
                                   model=params["model"], stop_details={"type": "refusal", "category": "cyber",
                                   "explanation": "This request was declined because it could enable cyber harm."})}
    amounts = [round(float(a) * 100) for a in re.findall(r"^- .*? (\d+\.\d\d)$", doc, re.M)]
    if SIM_TAGS["drops_line"] in doc and "Validation feedback" not in doc and amounts:
        amounts = amounts[:-1]                                     # a semantic slip, fixed by specific feedback
    total = re.search(r"^TOTAL: (\d+\.\d\d)$", doc, re.M)
    if total is None and "Amount paid" in system:                  # only a prompt that mentions it finds it
        total = re.search(r"^Amount paid: (\d+\.\d\d)$", doc, re.M)
    vendor = re.search(r"^Vendor: (.+)$", doc, re.M)
    number = re.search(r"^(?:INVOICE|RECEIPT|CREDIT NOTE) (\S+)$", doc, re.M)
    currency = re.search(r"^Currency: (\w+)$", doc, re.M)
    answer = json.dumps({"vendor": vendor and vendor.group(1), "invoice_number": number and number.group(1),
                         "currency": currency and currency.group(1), "line_items_cents": amounts,
                         "total_cents": total and round(float(total.group(1)) * 100)})
    n_out = approx_tokens(answer)
    if n_out > params["max_tokens"]:                                # cut off mid-JSON
        return {"message": message(answer[: params["max_tokens"] * 4], stop_reason="max_tokens",
                                   input_tokens=n_in, output_tokens=params["max_tokens"], model=params["model"])}
    return {"message": message(answer, input_tokens=n_in, output_tokens=n_out, model=params["model"])}


CORPUS = make_corpus()
print(len(CORPUS), "documents:", Counter(d["doc_type"] for d in CORPUS))
print("\n" + CORPUS[1]["text"])

## 7.7 Pre-flight: refine on a sample before the big run

Batch validation is **asynchronous**: a malformed request is reported hours later, and a bad prompt "succeeds" thousands of times. So before a large run: dry-run the request shape synchronously, refine the prompt on a representative sample until first-pass success is high, count tokens to chunk oversized documents **before** submission, and estimate the cost. [Theory](README.md#77-pre-flight-refine-on-a-sample-before-the-big-run)

### 1. Dry-run the request shape (offline)

In [ ]:
fake = FakeBatchAPI(fake_extractor, duration_minutes=45, context_limit=FAKE_CONTEXT_LIMIT)
api = fake.client
bad_params = {"model": MODEL, "max_tokens": 0, "system": "Extract the invoice as JSON.",
              "messages": [{"role": "user", "content": CORPUS[0]["text"]}]}

t0 = fake.now
try:
    api.messages.create(**bad_params)                        # synchronous: fails in seconds
except anthropic.BadRequestError as e:
    print(f"sync dry run   -> 400 after {(fake.now - t0).seconds} s of virtual time: {e.body['error']['message']}")

b = api.messages.batches.create(requests=[Request(custom_id="doc-0", params=bad_params)])
ended = wait_until_ended(api, b.id, sleep=fake.sleep)
err = next(iter(api.messages.batches.results(b.id))).result.error.error
print(f"same in a batch -> {err.type} discovered after {(fake.now - t0).seconds / 60:.0f} min: {err.message}")

### 2. Refine the prompt on a stratified sample (offline)

In [ ]:
class InvoiceFields(BaseModel):
    vendor: str
    invoice_number: str | None
    currency: str | None
    line_items_cents: list[int]
    total_cents: int | None


PROMPT_V1 = ("You extract data from finance documents. Return JSON with vendor, invoice_number, currency, "
             "line_items_cents (integer cents, one per line item) and total_cents (the TOTAL line, integer cents). "
             "Use null for anything not printed on the document.")
PROMPT_V2 = PROMPT_V1 + (" Document types: invoices and credit notes print 'TOTAL'; receipts print the total as "
                         "'Amount paid'. Treat that line as total_cents.")
OUTPUT_CONFIG = {"format": {"type": "json_schema", "schema": anthropic.transform_schema(InvoiceFields)}}


def extraction_params(text: str, system: str = PROMPT_V2, model: str = MODEL, max_tokens: int = 256) -> dict:
    return {"model": model, "max_tokens": max_tokens, "system": system, "output_config": OUTPUT_CONFIG,
            "messages": [{"role": "user", "content": f"<document>\n{text}\n</document>"}]}


def first_pass_ok(msg, doc_type: str) -> bool:
    if msg.stop_reason != "end_turn":
        return False
    inv = InvoiceFields.model_validate_json(msg.content[0].text)
    return inv.total_cents is not None and sum(inv.line_items_cents) == inv.total_cents


def stratified_sample(docs: list[dict], per_type: int = 4, seed: int = 1) -> list[dict]:
    rng, by_type = random.Random(seed), defaultdict(list)
    for d in docs:
        if len(d["text"]) < 2_000:                         # the sample is about the PROMPT, so skip giants here
            by_type[d["doc_type"]].append(d)
    return [d for t in sorted(by_type) for d in rng.sample(by_type[t], per_type)]


SAMPLE = stratified_sample(CORPUS)
sample_usage = []
for name, prompt in [("v1", PROMPT_V1), ("v2", PROMPT_V2)]:
    per_type, failed = defaultdict(list), []
    for d in SAMPLE:
        msg = api.messages.create(**extraction_params(d["text"], system=prompt))     # small, synchronous
        ok = first_pass_ok(msg, d["doc_type"])
        per_type[d["doc_type"]].append(ok)
        failed += [] if ok else [f"{d['doc_id']} ({msg.stop_reason})"]
        sample_usage.append(msg.usage)
    overall = sum(map(sum, per_type.values())) / len(SAMPLE)
    print(f"prompt {name}: first-pass {overall:.0%}  |  " +
          "  ".join(f"{t}: {sum(v)}/{len(v)}" for t, v in sorted(per_type.items())))
    print(f"           failed: {failed}")

> **Observe:** the aggregate hides *where* v1 fails: every receipt. Measuring **by document type** points straight at the fix (receipts print "Amount paid", not "TOTAL"). Twelve synchronous calls found what would otherwise have been 25% of a 50,000-document run. The one document that fails under both prompts has a different problem (print its text and its answer): the prompt is not the fix there, the pipeline's validation and retry are.

In [ ]:
for rate in (0.75, 0.85, 0.98):
    print(f"first-pass {rate:.0%} on 50,000 docs -> {round(50_000 * (1 - rate)):>6,} resubmissions "
          f"(each another batch cycle of up to 24 h)")

### 3. Count tokens and chunk oversized documents before submission (offline)

In [ ]:
def doc_tokens(api: anthropic.Anthropic, text: str, system: str = PROMPT_V2) -> int:
    """messages.count_tokens is free (rate-limited) and uses the real tokenizer on the live API."""
    return api.messages.count_tokens(model=MODEL, system=system,
                                     messages=[{"role": "user", "content": f"<document>\n{text}\n</document>"}]).input_tokens


def chunk_document(text: str, max_tokens_per_chunk: int = 800) -> list[str]:
    """Split by line items, repeating the 3 header lines in every chunk so each chunk is self-describing."""
    lines = text.splitlines()
    header, body = lines[:3], lines[3:]
    chunks, current = [], []
    for line in body:
        if current and approx_tokens("\n".join(header + current + [line])) > max_tokens_per_chunk:
            chunks.append(current)
            current = []
        current.append(line)
    chunks.append(current)
    return ["\n".join(header + [f"(part {i + 1} of {len(chunks)})"] + c) for i, c in enumerate(chunks)]


oversized = [(d["doc_id"], n) for d in CORPUS if (n := doc_tokens(api, d["text"])) > FAKE_CONTEXT_LIMIT]
print("over the (scaled) context window:", oversized)
parts = chunk_document(next(d for d in CORPUS if d["doc_id"] == oversized[0][0])["text"])
print(f"{oversized[0][0]} -> {len(parts)} chunks of", [approx_tokens(p) for p in parts], "tokens (approx)")
print("custom_ids:", [make_custom_id(oversized[0][0], chunk=i) for i in range(len(parts))])

### 4. Estimate the cost from the sample's usage (offline)

In [ ]:
avg_in = sum(u.input_tokens for u in sample_usage) / len(sample_usage)
avg_out = sum(u.output_tokens for u in sample_usage) / len(sample_usage)
print(f"sample: avg {avg_in:.0f} input / {avg_out:.0f} output tokens per document (fake tokenizer)")
print("-> plug these into estimate_cost() in 7.9 and get sign-off before spending at scale")

> **Try it:** write a `PROMPT_V3` that also asks for credit-note amounts as negative cents, re-run the sample, and watch the semantic check (sum of lines == total) still pass. Then add a doc type the sample does not cover and see why the sample must be **stratified**.

## 7.8 SLA planning and submission cadence

Plan from the **24-hour upper bound**, not from the typical hour: worst-case latency = window `W` (longest wait before submission) + processing `P` (24 h) + buffer `B` (validation, retries, downstream). So `W ≤ SLA − P − B`. A 30-hour SLA with a 2-hour buffer gives 4-hour windows. [Theory](README.md#78-sla-planning-and-submission-cadence)

In [ ]:
def max_window(sla_h: float, processing_h: float = 24, buffer_h: float = 2, batch_retry_cycles: int = 0) -> float:
    """Longest submission window that still meets the SLA in the worst case (<= 0 means: not feasible)."""
    return sla_h - processing_h * (1 + batch_retry_cycles) - buffer_h


def latest_submit(deadline: datetime, processing_h: float = 24, buffer_h: float = 2) -> datetime:
    return deadline - timedelta(hours=processing_h + buffer_h)


for sla, retries in [(30, 0), (36, 0), (48, 0), (26, 0), (54, 1), (30, 1)]:
    w = max_window(sla, batch_retry_cycles=retries)
    verdict = f"submit every {w:g} h" if w > 0 else "NOT feasible with batches: use sync for (some of) it"
    print(f"SLA {sla:>2} h, {retries} batch retry cycle(s): {verdict}")

monday_8am = datetime(2026, 10, 5, 8, 0)
print("\nReport due", monday_8am.strftime("%a %H:%M"), "-> submit by", latest_submit(monday_8am).strftime("%a %H:%M"))

### Simulate continuous arrivals against a 30-hour SLA (offline)

In [ ]:
def worst_age_hours(window_h: float, processing_h: float, horizon_h: float = 72, every_min: int = 10) -> float:
    """Documents arrive every `every_min` minutes; a batch is submitted at each multiple of window_h."""
    worst = 0.0
    for k in range(int(horizon_h * 60 / every_min)):
        arrival = k * every_min / 60
        submit = math.ceil(arrival / window_h) * window_h
        worst = max(worst, submit - arrival + processing_h)
    return worst


SLA, BUFFER = 30, 2
print(f"{'window':>7} {'typical (1 h)':>14} {'worst case (24 h)':>18}   meets {SLA} h SLA with {BUFFER} h buffer?")
for w in (1, 4, 6, 8, 12):
    typical, worst = worst_age_hours(w, 1), worst_age_hours(w, 24)
    print(f"{w:>5} h {typical:>12.1f} h {worst:>16.1f} h   {'yes' if worst + BUFFER <= SLA else 'NO'}")

> **Observe:** every window looks fine if you plan from the typical one-hour batch. Only the worst case separates 4-hour windows (28 h + 2 h buffer = 30 h) from 6-hour windows (30 h with zero buffer). Retries that must land inside a tight SLA go **synchronous**, because another batch cycle can add 24 hours.

> **Try it:** a team insists on batch-priced retries. What is the shortest SLA that allows one batch retry cycle with 4-hour windows and a 2-hour buffer? Check your answer with `max_window`.

## 7.9 Cost modeling: batch discount, prompt caching and model choice

Batch prices are half of standard. Prompt caching **stacks** on top (5-minute write 1.25×, 1-hour write 2×, read 0.1× base on most models, 0.05× on Opus 5.5), but batch cache hits are best-effort (typically 30–98%). A prefix below the model's minimum cacheable length is silently not cached. [Theory](README.md#79-cost-modeling-batch-discount-prompt-caching-and-model-choice)

In [ ]:
# Standard prices, USD per million tokens (Claude API, October 2026). Batch = 50% of these.
PRICES = {
    "claude-sonnet-5-5": {"in": 2.00, "out": 10.00, "read_mult": 0.10, "min_cache": 512},
    "claude-haiku-4-5":  {"in": 1.00, "out": 5.00,  "read_mult": 0.10, "min_cache": 4_096},
    "claude-opus-5-5":   {"in": 4.00, "out": 20.00, "read_mult": 0.05, "min_cache": 512},
}
WRITE_MULT = {"5m": 1.25, "1h": 2.0}


def price_of(model: str) -> dict:
    return PRICES.get(model, PRICES["claude-sonnet-5-5"])   # unknown model id: assume Sonnet 5.5 prices


def estimate_cost(n_docs: int, prefix_tokens: int, doc_tokens: int, output_tokens: int, model: str = MODEL,
                  batch: bool = True, cache_hit_rate: float = 0.0, ttl: str = "5m") -> float:
    p, disc = price_of(model), 0.5 if batch else 1.0
    base_in, out = p["in"] * disc / 1e6, p["out"] * disc / 1e6
    cacheable = cache_hit_rate > 0 and prefix_tokens >= p["min_cache"]
    if cacheable:
        reads = n_docs * cache_hit_rate * prefix_tokens
        writes = n_docs * (1 - cache_hit_rate) * prefix_tokens
        prefix_cost = reads * base_in * p["read_mult"] + writes * base_in * WRITE_MULT[ttl]
    else:
        prefix_cost = n_docs * prefix_tokens * base_in
    return prefix_cost + n_docs * doc_tokens * base_in + n_docs * output_tokens * out


SCENARIO = dict(n_docs=10_000, prefix_tokens=3_000, doc_tokens=2_000, output_tokens=500)
rows = [
    ("Sonnet 5.5, synchronous, no caching", dict(batch=False)),
    ("Sonnet 5.5, batch, no caching", dict()),
    ("Sonnet 5.5, batch + 5-min cache, 70% hits", dict(cache_hit_rate=0.7)),
    ("Sonnet 5.5, batch + 1-hour cache, 95% hits", dict(cache_hit_rate=0.95, ttl="1h")),
    ("Haiku 4.5, batch + 'caching' (prefix < 4,096)", dict(model="claude-haiku-4-5", cache_hit_rate=0.7)),
    ("Opus 5.5, batch + 5-min cache, 70% hits", dict(model="claude-opus-5-5", cache_hit_rate=0.7)),
]
for label, kw in rows:
    print(f"{label:48} ${estimate_cost(**SCENARIO, **kw):8.2f}")

> **Observe:** the first rows reproduce the README's worked example ($150 → $75 → ≈ $58). The Haiku row shows the silent trap: its 3,000-token prefix is below Haiku 4.5's 4,096-token minimum, so the "caching" saves nothing (still $37.50).

### Measure, don't trust the estimate: cost from `usage` (offline)

In [ ]:
def actual_cost(usages: list, model: str, batch: bool = True, ttl: str = "5m") -> float:
    """Sum real usage fields. input_tokens counts only the uncached part of the prompt."""
    p, disc = price_of(model), 0.5 if batch else 1.0
    base_in, out = p["in"] * disc / 1e6, p["out"] * disc / 1e6
    total = 0.0
    for u in usages:
        total += u.input_tokens * base_in + u.output_tokens * out
        total += (u.cache_creation_input_tokens or 0) * base_in * WRITE_MULT[ttl]
        total += (u.cache_read_input_tokens or 0) * base_in * p["read_mult"]
    return total


Usage = anthropic.types.Usage
usages = [Usage(input_tokens=2_000, output_tokens=480, cache_creation_input_tokens=3_000, cache_read_input_tokens=0)] + \
         [Usage(input_tokens=2_000, output_tokens=510, cache_creation_input_tokens=0, cache_read_input_tokens=3_000)] * 9
print(f"10 results: ${actual_cost(usages, MODEL):.4f} at batch prices "
      f"(${actual_cost(usages, MODEL, batch=False):.4f} if run synchronously)")
hit_rate = sum(u.cache_read_input_tokens > 0 for u in usages) / len(usages)
print(f"observed cache hit rate: {hit_rate:.0%}")

### The documented recipe for better batch cache hits (offline)

In [ ]:
FEW_SHOT = "Example document ... Example JSON ...\n" * 120          # a long, identical shared block
SHARED_SYSTEM = [
    {"type": "text", "text": PROMPT_V2},
    {"type": "text", "text": FEW_SHOT, "cache_control": {"type": "ephemeral", "ttl": "1h"}},
]


def cached_request(cid: str, text: str) -> Request:
    return Request(custom_id=cid, params={"model": MODEL, "max_tokens": 256, "system": SHARED_SYSTEM,
                                          "messages": [{"role": "user", "content": f"<document>\n{text}\n</document>"}]})


all_requests = [cached_request(make_custom_id(d["doc_id"]), d["text"]) for d in CORPUS[:20]]
warm_batch, rest = all_requests[:1], all_requests[1:]       # 1) one request writes the 1-hour cache entry

prefix_hashes = {hashlib.sha256(json.dumps(r["params"]["system"]).encode()).hexdigest() for r in all_requests}
prefix_tokens = approx_tokens(PROMPT_V2 + FEW_SHOT)
print("identical prefix in every request:", len(prefix_hashes) == 1)
print(f"prefix ~{prefix_tokens} tokens; cacheable on {MODEL}: {prefix_tokens >= price_of(MODEL)['min_cache']}; "
      f"on claude-haiku-4-5: {prefix_tokens >= PRICES['claude-haiku-4-5']['min_cache']}")
print(f"plan: submit {len(warm_batch)} warm-up request, wait for it to end, then submit the other {len(rest)}")

> **Try it:** move the document *before* the few-shot block in `cached_request` and recompute `prefix_hashes`. Every request now has a different prefix, so nothing can be shared. Order matters: stable content first, varying content last.

## 7.10 Extended output (beta) on batches

The beta header `output-300k-2026-03-24` raises `max_tokens` to **300,000** for batch requests on Opus 5.5, Opus 5, Opus 4.8, Opus 4.7, Opus 4.6, Sonnet 5.5, Sonnet 5 and Sonnet 4.6. The synchronous cap stays at 128K. It is available only on the Message Batches API (Claude API and Claude Platform on AWS). [Theory](README.md#710-extended-output-beta-on-batches)

In [ ]:
from anthropic.types.beta.message_create_params import MessageCreateParamsNonStreaming as BetaParams
from anthropic.types.beta.messages.batch_create_params import Request as BetaRequest

guide = BetaParams(model=MODEL, max_tokens=300_000,
                   messages=[{"role": "user", "content": "Write the complete developer guide for our billing API ..."}])
print("without the beta:", batch_param_problems(guide))
print("with the beta:   ", batch_param_problems(guide, betas=(EXTENDED_OUTPUT_BETA,)) or "OK")
print("on Haiku 4.5:    ", batch_param_problems({**guide, "model": "claude-haiku-4-5"}, betas=(EXTENDED_OUTPUT_BETA,)))

fake = FakeBatchAPI(duration_minutes=90)
b = fake.client.beta.messages.batches.create(
    betas=[EXTENDED_OUTPUT_BETA],
    requests=[BetaRequest(custom_id="api-guide-001", params=guide)],
)
print("\nSDK class:", type(b).__name__, "| anthropic-beta header sent:", fake.batches[b.id].headers.get("anthropic-beta"))

> **Observe:** the SDK's beta namespace (`client.beta.messages.batches`) sends the `anthropic-beta` header for you. A single 300K-token generation can take over an hour, so leave room inside the 24-hour window, and check `stop_reason` for `max_tokens` exactly as you would for a short request.

## 7.11 Operations: rate limits, retention, privacy and platforms

The Batches API has its own limits (requests per minute to the batch endpoints, and a cap on batch requests waiting in the processing queue), separate from Messages API limits. Data is stored for up to 29 days after creation, so batches are **not ZDR-eligible** (and not HIPAA-eligible). The `/v1/messages/batches` endpoint is not supported on Amazon Bedrock, Google Cloud Vertex AI or Microsoft Foundry. Bedrock (batch inference) and Vertex AI (batch predictions) have their own batch services for Claude, with their own formats, limits and model lists; Microsoft Foundry documents no batch option for Claude, so there the choice is synchronous calls behind your own queue, or running the batch work on the Claude API. [Theory](README.md#711-operations-rate-limits-retention-privacy-and-platforms)

In [ ]:
TIERS = {"start": {"rpm": 1_000, "queue": 200_000}, "build": {"rpm": 2_000, "queue": 300_000},
         "scale": {"rpm": 4_000, "queue": 500_000}}


def polling_rpm(n_batches: int, interval_s: float) -> float:
    return n_batches * 60 / interval_s


def queue_headroom(tier: str, waiting: int, new: int) -> int:
    return TIERS[tier]["queue"] - waiting - new


print("200 batches polled every 10 s:", polling_rpm(200, 10), "RPM vs Start tier", TIERS["start"]["rpm"])
print("200 batches with 10-min backoff:", polling_rpm(200, 600), "RPM")
for new in (50_000, 150_000):
    h = queue_headroom("start", waiting=120_000, new=new)
    print(f"queue: 120,000 waiting + {new:,} new on Start -> {'fits' if h >= 0 else f'over by {-h:,}: wait or split over time'}")

In [ ]:
def retention(created_at: datetime) -> dict:
    return {"results_downloadable_until": created_at + timedelta(days=29),
            "expires_at (processing)": created_at + timedelta(hours=24)}


created = datetime(2026, 10, 3, 8, 0, tzinfo=timezone.utc)
for k, v in retention(created).items():
    print(f"{k:28} {v:%Y-%m-%d %H:%M} UTC")


class BatchBackend(Protocol):
    """Keep the pipeline independent of where batches run (Claude API, or a cloud provider's batch service)."""
    def submit(self, requests: list[Request]) -> str: ...
    def status(self, batch_id: str) -> str: ...
    def results(self, batch_id: str): ...


def choose_backend(platform: str, zdr_required: bool) -> str:
    if zdr_required:
        return "synchronous Messages API (Batches API is not ZDR-eligible)"
    if platform == "bedrock":
        return "Bedrock batch inference (check that your Claude model and Region are listed)"
    if platform == "vertex":
        return "Vertex AI batch predictions for Claude (input from Cloud Storage or BigQuery)"
    if platform == "foundry":
        return "no documented Claude batch option: synchronous calls behind your own queue, or the Claude API"
    return "Message Batches API (/v1/messages/batches)"


for platform, zdr in [("claude-api", False), ("claude-api", True), ("bedrock", False), ("vertex", False), ("foundry", False)]:
    print(f"{platform:11} ZDR={zdr!s:5} -> {choose_backend(platform, zdr)}")

> **Observe:** polling is the cheapest place to blow the batch RPM budget; backoff fixes it. Retention is measured from `created_at`, not from when processing ended: delete batches promptly once results are safely ingested if the customer's policy needs it.

> **Try it:** implement `AnthropicBatches(BatchBackend)` as a thin wrapper around `client.messages.batches`, and a `FakeBackend` around `FakeBatchAPI`. The mini-project below takes the SDK client directly; swapping backends is the production version of the same idea.

## 7.12 Reliability around a batch pipeline (Domain 5)

The API runs requests; reliability lives in the code around it: a manifest for crash recovery, reconciliation so nothing is silently lost, structured failure records, idempotent sinks keyed by `custom_id`, coverage reports by document type, and human review for what cannot be resolved. [Theory](README.md#712-reliability-around-a-batch-pipeline-domain-5)

In [ ]:
def reconcile(manifest_ids: list[str], result_ids: list[str]) -> dict[str, list[str]]:
    seen = Counter(result_ids)
    return {"missing": sorted(set(manifest_ids) - set(seen)),
            "unexpected": sorted(set(seen) - set(manifest_ids)),
            "duplicated": sorted(k for k, n in seen.items() if n > 1)}


print(reconcile(["a", "b", "c", "d"], ["b", "a", "d", "d", "x"]))


@dataclass
class FailureRecord:                  # what an on-call engineer needs, not just "failed"
    custom_id: str
    failure_type: str
    detail: str
    attempt: int
    model: str
    next_action: str


class IdempotentSink:
    """Upsert by custom_id: replays and resubmissions never duplicate rows."""
    def __init__(self):
        self.rows: dict[str, dict] = {}

    def upsert(self, custom_id: str, row: dict) -> None:
        self.rows[custom_id] = row


sink = IdempotentSink()
for _ in range(2):                    # the same results file processed twice (e.g. after a crash)
    for cid in ("inv-1", "inv-2", "inv-3"):
        sink.upsert(cid, {"status": "accepted"})
print("rows after replaying twice:", len(sink.rows))
print(FailureRecord("inv-2026-000417", "errored", "prompt is too long", 1, MODEL, "chunk into 4 parts, resubmit"))

In [ ]:
def coverage_by_type(outcomes: list[tuple[str, str]]) -> None:
    """outcomes: (doc_type, final_status). Aggregates hide weak segments; print both."""
    by_type = defaultdict(Counter)
    for t, s in outcomes:
        by_type[t][s] += 1
    total = Counter(s for _, s in outcomes)
    print(f"ALL          accepted {total['accepted'] / len(outcomes):.0%}  {dict(total)}")
    for t, c in sorted(by_type.items()):
        print(f"{t:12} accepted {c['accepted'] / sum(c.values()):.0%}  {dict(c)}")


demo = [("invoice", "accepted")] * 880 + [("invoice", "human")] * 10 + [("credit_note", "accepted")] * 60 + \
       [("credit_note", "human")] * 40 + [("receipt", "accepted")] * 10
coverage_by_type(demo)

> **Observe:** a 95% aggregate hides credit notes at 60%. Report by segment before anyone proposes to cut human review.

## Mini-project: a batch extraction pipeline with resubmission, review and reporting

Everything above in one pipeline over the 100-document `CORPUS`:

1. **Manifest** (7.4, 7.12): one ledger row per `custom_id`, persisted to disk after every state change.
2. **Pre-flight** (7.7): prompt v2 (chosen on the sample), token counts, and **chunking of oversized documents before submission** (toggle with `preflight_chunking`).
3. **Submit → poll → stream** (7.1, 7.5): crash-safe state, backoff polling, results matched by `custom_id`.
4. **Triage** (7.6): `classify()` plus Pydantic schema validation and the semantic check (lines sum to the total).
5. **Resubmit only the failures, with modifications** (7.6): chunk, raise `max_tokens`, add validation feedback, switch refusals to `FALLBACK_MODEL`, resend transient failures unchanged; cap attempts; go **synchronous** when the SLA no longer has room for another batch cycle (7.8).
6. **Human-review queue** for what a retry cannot fix (missing data, exhausted attempts) (7.12).
7. **Coverage and cost report** by document type, from real `usage` fields (7.9, 7.12).

It runs offline on `FakeBatchAPI` by default. Set `LIVE = True` to run the **same code** against the real API (then the failures are whatever the real model produces).

In [ ]:
@dataclass
class Item:                                    # one ledger row = one custom_id
    custom_id: str
    doc_id: str
    doc_type: str
    text: str
    parent: str | None = None                  # set on chunks
    model: str = MODEL
    max_tokens: int = 256
    feedback: str | None = None
    prev_answer: str | None = None
    attempt: int = 0
    sync_attempts: int = 0
    status: str = "pending"                    # pending | submitted | retry | accepted | split | human
    reason: str = ""
    batch_ids: list[str] = field(default_factory=list)
    data: dict | None = None


class BatchPipeline:
    def __init__(self, api: anthropic.Anthropic, *, sleep: Callable[[float], None], clock: Callable[[], datetime],
                 workdir: pathlib.Path, system: str = PROMPT_V2, preflight_chunking: bool = True,
                 token_limit: int = FAKE_CONTEXT_LIMIT, sla_hours: float = 30, buffer_hours: float = 2,
                 max_attempts: int = 3, poll_first_delay: float = 30):
        self.api, self.sleep, self.clock, self.workdir = api, sleep, clock, workdir
        self.system, self.preflight_chunking, self.token_limit = system, preflight_chunking, token_limit
        self.sla_hours, self.buffer_hours, self.max_attempts = sla_hours, buffer_hours, max_attempts
        self.poll_first_delay = poll_first_delay
        self.items: dict[str, Item] = {}
        self.spend: list[tuple[str, str, Any]] = []        # (mode, model, usage)
        self.events: list[str] = []
        self.started = clock()

    # ---------------- manifest
    def save(self) -> None:
        state = {cid: {k: v for k, v in vars(it).items() if k != "text"} for cid, it in self.items.items()}
        (self.workdir / "manifest.json").write_text(json.dumps(state, default=str))

    def add_docs(self, docs: list[dict]) -> None:
        for d in docs:
            cid = make_custom_id(d["doc_id"])
            self.items[cid] = Item(cid, d["doc_id"], d["doc_type"], d["text"])
        self.save()

    # ---------------- pre-flight
    def split(self, it: Item, reason: str) -> list[Item]:
        it.status, it.reason = "split", reason
        chunks = []
        for i, part in enumerate(chunk_document(it.text)):
            cid = make_custom_id(it.doc_id, chunk=i)
            chunks.append(Item(cid, it.doc_id, it.doc_type, part, parent=it.custom_id, model=it.model))
            self.items[cid] = chunks[-1]
        return chunks

    def preflight(self) -> None:
        if not self.preflight_chunking:
            return
        for it in list(self.items.values()):
            n = doc_tokens(self.api, it.text, self.system)
            if n > self.token_limit:
                parts = self.split(it, f"pre-flight: {n} tokens > {self.token_limit}")
                self.events.append(f"pre-flight: {it.doc_id} ({n} tokens) split into {len(parts)} chunks")
        self.save()

    # ---------------- requests
    def params_for(self, it: Item) -> dict:
        p = extraction_params(it.text, system=self.system, model=it.model, max_tokens=it.max_tokens)
        if it.feedback:                                    # Chapter 6 retry-with-feedback, as a pre-built history
            p["messages"] += [{"role": "assistant", "content": it.prev_answer or ""},
                              {"role": "user", "content": f"Validation feedback: {it.feedback} Return corrected JSON."}]
        return p

    def submit_batch(self, items: list[Item]) -> str:
        batch = self.api.messages.batches.create(
            requests=[Request(custom_id=it.custom_id, params=self.params_for(it)) for it in items])
        for it in items:                                    # persist the batch id before anything else
            it.status, it.attempt = "submitted", it.attempt + 1
            it.batch_ids.append(batch.id)
        self.save()
        return batch.id

    # ---------------- triage
    def handle(self, it: Item, bucket: str, reason: str, parsed: Any, msg=None, mode: str = "batch") -> None:
        if msg is not None:
            self.spend.append((mode, it.model, msg.usage))
        it.reason = reason
        if bucket == "ok":
            it.status, it.data = "accepted", parsed.model_dump()
        elif bucket == "human":
            it.status = "human"
        elif it.attempt >= self.max_attempts:
            it.status, it.reason = "human", f"gave up after {it.attempt} attempts: {reason}"
        else:
            it.status = "retry"
            if bucket == "fallback_model":
                it.model = FALLBACK_MODEL
            elif "prompt is too long" in reason:
                self.split(it, reason)
            elif reason.startswith("truncated"):
                it.max_tokens *= 2
            elif bucket == "fix_then_retry":
                it.feedback = reason
                it.prev_answer = "".join(b.text for b in msg.content if b.type == "text") if msg else None

    def semantic(self, it: Item):
        return None if it.parent else invoice_semantics_fields   # chunks are checked after merging

    def collect(self, batch_id: str) -> None:
        submitted = [cid for cid, it in self.items.items() if it.batch_ids[-1:] == [batch_id]]
        seen, types, buckets = [], Counter(), Counter()
        for res in self.api.messages.batches.results(batch_id):
            it = self.items[res.custom_id]
            seen.append(res.custom_id)
            bucket, reason, parsed = classify(res, InvoiceFields, self.semantic(it))
            msg = res.result.message if res.result.type == "succeeded" else None
            self.handle(it, bucket, reason, parsed, msg)
            types[res.result.type] += 1
            buckets[bucket] += 1
        self.events.append(f"   results {dict(types)} -> triage {dict(buckets)}")
        problems = reconcile(submitted, seen)
        if any(problems.values()):
            self.events.append(f"RECONCILIATION ALERT {batch_id}: {problems}")
        self.save()

    def run_sync(self, items: list[Item]) -> None:
        for it in items:
            it.attempt += 1
            it.sync_attempts += 1
            try:
                msg = self.api.messages.create(**self.params_for(it))
            except anthropic.APIStatusError as e:
                self.handle(it, "retry_as_is" if e.status_code >= 500 else "fix_then_retry", str(e.message), None)
                continue
            fake_item = MessageBatchIndividualResponse.model_validate(
                {"custom_id": it.custom_id, "result": {"type": "succeeded", "message": msg.model_dump()}})
            bucket, reason, parsed = classify(fake_item, InvoiceFields, self.semantic(it))
            self.handle(it, bucket, reason, parsed, msg, mode="sync")
        self.save()

    # ---------------- merge chunks back into documents
    def merge_chunks(self) -> None:
        for parent in [it for it in self.items.values() if it.status == "split"]:
            chunks = [c for c in self.items.values() if c.parent == parent.custom_id]
            if any(c.status in ("human",) for c in chunks):
                parent.status, parent.reason = "human", "a chunk could not be extracted"
            elif all(c.status == "accepted" for c in chunks):
                merged = {"vendor": chunks[0].data["vendor"], "invoice_number": chunks[0].data["invoice_number"],
                          "currency": chunks[0].data["currency"],
                          "line_items_cents": [x for c in chunks for x in c.data["line_items_cents"]],
                          "total_cents": next((c.data["total_cents"] for c in chunks if c.data["total_cents"]), None)}
                verdict = invoice_semantics_fields(InvoiceFields(**merged))
                parent.status = "accepted" if verdict is None else "human"
                parent.reason = "merged from chunks" if verdict is None else f"merged: {verdict[1]}"
                parent.data = merged

    # ---------------- main loop
    def hours_left(self) -> float:
        return self.sla_hours - (self.clock() - self.started).total_seconds() / 3600

    def run(self, docs: list[dict]) -> None:
        self.add_docs(docs)
        self.preflight()
        cycle = 0
        while True:
            todo = [it for it in self.items.values() if it.status in ("pending", "retry")]
            if not todo:
                break
            cycle += 1
            if self.hours_left() >= 24 + self.buffer_hours:          # worst case still fits the SLA (7.8)
                bid = self.submit_batch(todo)
                self.events.append(f"cycle {cycle}: batch {bid} with {len(todo)} requests "
                                   f"({self.hours_left():.1f} h of SLA left)")
                if wait_until_ended(self.api, bid, sleep=self.sleep, first_delay=self.poll_first_delay) is None:
                    self.events.append(f"gave up waiting for {bid}; manifest.json has its id, resume later")
                    break
                self.collect(bid)
            else:
                self.events.append(f"cycle {cycle}: only {self.hours_left():.1f} h left -> {len(todo)} SYNC calls")
                self.run_sync(todo)
        self.merge_chunks()
        self.save()


def invoice_semantics_fields(inv: InvoiceFields):
    if inv.total_cents is None:
        return "human", "total not in the document (null): route to review, do not retry"
    if sum(inv.line_items_cents) != inv.total_cents:
        return "fix_then_retry", (f"line items sum to {sum(inv.line_items_cents)} but total_cents is "
                                  f"{inv.total_cents}. Re-read every line item.")
    return None


print("pipeline defined")

### Reporting

In [ ]:
def report(p: BatchPipeline) -> dict:
    docs = {it.custom_id: it for it in p.items.values() if it.parent is None}
    print("\n".join(p.events))
    print(f"\n{'doc type':12} {'docs':>5} {'1st pass':>9} {'after retry':>12} {'human':>6}")
    by_type = defaultdict(Counter)
    delayed = 0
    for it in docs.values():
        family = [it] + [c for c in p.items.values() if c.parent == it.custom_id]
        rounds = len({b for x in family for b in x.batch_ids}) + max(x.sync_attempts for x in family)
        delayed += rounds > 1
        key = "human" if it.status == "human" else "first" if rounds == 1 else "retry"
        by_type[it.doc_type][key] += 1
    for t, c in sorted(by_type.items()):
        print(f"{t:12} {sum(c.values()):>5} {c['first']:>9} {c['retry']:>12} {c['human']:>6}")
    human = [(it.doc_id, it.reason) for it in docs.values() if it.status == "human"]
    print("\nHuman-review queue:", *[f"  {d}: {r}" for d, r in human], sep="\n")
    cost = sync_equiv = 0.0
    for mode, model, u in p.spend:
        price = price_of(model)
        standard = (u.input_tokens * price["in"] + u.output_tokens * price["out"]) / 1e6
        cost += standard * (0.5 if mode == "batch" else 1.0)
        sync_equiv += standard
    accepted = sum(it.status == "accepted" for it in docs.values())
    print(f"\nAccepted {accepted}/{len(docs)} documents | requests billed: {len(p.spend)} | "
          f"cost ${cost:.4f} (all-synchronous: ${sync_equiv:.4f}) | ${cost / max(accepted, 1):.6f} per accepted doc")
    print("(fake tokenizer and a scaled-down world: compare ratios, not absolute dollars)")
    sent = sum(len(x.batch_ids) + x.sync_attempts for x in p.items.values())
    return {"accepted": accepted, "docs": len(docs), "cycles": sum(e.startswith("cycle") for e in p.events),
            "sent": sent, "delayed": delayed, "cost": cost}

### Run A: the exam scenario (no pre-flight chunking)

In [ ]:
LIVE = False                       # True: same pipeline, real API, real time (costs money; see the intro)


def make_pipeline(**kw) -> BatchPipeline:
    workdir = pathlib.Path(tempfile.mkdtemp(prefix="ch07-pipeline-"))
    if LIVE:
        return BatchPipeline(client, sleep=time.sleep, clock=lambda: datetime.now(timezone.utc),
                             workdir=workdir, poll_first_delay=60, **kw)
    sim = FakeBatchAPI(fake_extractor, duration_minutes=55, context_limit=FAKE_CONTEXT_LIMIT, seed=3)
    return BatchPipeline(sim.client, sleep=sim.sleep, clock=lambda: sim.now, workdir=workdir, **kw)


run_a = make_pipeline(preflight_chunking=False)
run_a.run(CORPUS)
summary_a = report(run_a)

> **Observe:** cycle 2 contains **only** the failed `custom_id`s, each changed according to its failure: the three oversized documents come back as chunk requests (`...-c0`, `-c1`, ...), the truncated ones with doubled `max_tokens`, the dropped-line ones with validation feedback as a pre-built multi-turn history, the refused one on `FALLBACK_MODEL`, and the overloaded/expired ones unchanged. The two invoices with no total go to humans without a retry, because the information is not in the source.

### Run B: with pre-flight chunking, and the comparison

In [ ]:
run_b = make_pipeline(preflight_chunking=True)
run_b.run(CORPUS)
summary_b = report(run_b)

print(f"\n{'':18} {'cycles':>6} {'requests sent':>14} {'docs needing a 2nd cycle':>25} {'accepted':>9} {'cost':>9}")
for name, s in [("A (no pre-flight)", summary_a), ("B (pre-flight)", summary_b)]:
    print(f"{name:18} {s['cycles']:>6} {s['sent']:>14} {s['delayed']:>25} {s['accepted']:>5}/{s['docs']} ${s['cost']:.4f}")

> **Observe:** the cost is the same, because `errored` requests are not billed. The price of skipping pre-flight is **time**: in Run A the three oversized documents error in cycle 1 and wait for a second batch cycle (up to another 24 hours on a real run); in Run B they are chunked before submission and finish in cycle 1. Every failure class you remove before submission saves a cycle for those documents.

> **Try it:**
> 1. Run with `sla_hours=26.5`: cycle 1 still fits (24 h + 2 h buffer), but after it there is no room for another batch cycle, so the retries switch to **synchronous** calls at full price (check the events and the cost). With `sla_hours=20` nothing is batched at all.
> 2. Run with `system=PROMPT_V1` and look at the receipts row and the human-review queue. This is what skipping the sample run costs.
> 3. Set `max_attempts=1` and confirm nothing is retried: every failure lands in the human queue with a reason.
> 4. Kill the run halfway (raise an exception inside `collect` after 50 results), then write a `resume()` that reloads `manifest.json` and continues polling the saved batch instead of resubmitting.
> 5. With a key and credit, set `LIVE = True` and run Run B only. Compare the human-review queue with the offline one.

## Recap

- A batch = many independent Messages requests, each with your `custom_id`; **50% off** every token, **no latency SLA** (up to 24 h, most within 1 h), 100,000 requests or 256 MB per batch, results for 29 days after creation.
- Choose by "is anyone blocked?": blocking work stays synchronous; scheduled work goes to batches if the SLA covers 24 h + buffer and no client tool must run mid-request. Server tools work inside batches; `pause_turn` is continued with a follow-up request.
- `custom_id` is the primary key: derived from your record key, legal, PII-free, stable across retries, `-cN` for chunks; details live in a ledger. Never match by position.
- Persist the batch ID at submit time; poll with backoff; `succeeded = 0` while `in_progress` means nothing; stream results; cancel ends the batch normally; delete only after it ends.
- Four result types, and **succeeded ≠ usable**: check `stop_reason` (`refusal`, `max_tokens`, `model_context_window_exceeded`, `pause_turn`, `tool_use`), validate schema and semantics. Resubmit only the failures, each with its fix; route missing information to humans.
- Pre-flight: synchronous dry run, sample-based prompt refinement by document type, token counting and chunking, cost estimate.
- SLA math from the 24-hour bound (`W ≤ SLA − 24 − B`); retries inside a tight SLA go synchronous.
- Cost: batch × caching × model tier compound; check minimum cacheable lengths; measure with `usage`.
- Operations: separate batch rate limits, 29-day retention (not ZDR- or HIPAA-eligible), extended output to 300K via beta. `/v1/messages/batches` is not on Bedrock, Vertex AI or Foundry: use Bedrock batch inference or Vertex AI batch predictions, and on Foundry (no documented batch option) a synchronous queue.

Next: [homework](02_homework.ipynb)